# Spider runner (Group 3)
Runs the Spider notebook's steps in the background, without the viewer cells. Runtime: GPU. Then **Run all**. Paste your W&B key when cell 2 asks.

In [ ]:
import os, sys
os.environ["MUJOCO_GL"] = "egl"
!nvidia-smi -L
if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q https://github.com/ma-xr/pupper-mjlab.git /content/pupper-mjlab
%cd /content/pupper-mjlab
!git pull -q
!pip install -q -e .
print("mjlab ready")

In [ ]:
import wandb
wandb.login()  # paste your key from https://wandb.ai/authorize when asked
print("logged in as", wandb.Api().default_entity)

In [ ]:
# Stage 1: clip policy (spider_down / spider_idle / spider_up), in the background.
!cd /content/pupper-mjlab && git pull -q && STAGE=clips ITERATIONS=1000 nohup python -u notebooks/run_spider.py > /content/spider_clips.log 2>&1 < /dev/null &
print("clip training started; check progress with the next cell")

In [ ]:
# Progress (re-run any time)
!grep -aE "ok |FAIL|verdict|worst|WOBBLES|FALLS|HOLDS|clip |Learning iteration|ETA|Error|Traceback" /content/spider_clips.log | tail -25
!grep -aE "Learning iteration|ETA|Error|Traceback" /content/spider_walk.log 2>/dev/null | tail -5

In [ ]:
# Stage 2: spider walk. Run this cell by hand once the clip policy is done (one training at a time on the T4).
!cd /content/pupper-mjlab && git pull -q
clips_done = "score the clip policy" in open("/content/spider_clips.log").read() if os.path.exists("/content/spider_clips.log") else False
if clips_done:
  !cd /content/pupper-mjlab && STAGE=walk nohup python -u notebooks/run_spider.py > /content/spider_walk.log 2>&1 < /dev/null &
  print("walk training started")
else:
  print("clip training not finished yet: run this cell again later")